Now we get to more detail:

Different models

Structured Outputs

Guardrails

In [2]:
# Import necessary libraries and modules for working with OpenAI and agents
from dotenv import load_dotenv # Load environment variables from a .env file
from openai import AsyncOpenAI # Import the AsyncOpenAI client for interacting with OpenAI
from agents import Agent, Runner, trace, set_tracing_disabled, set_tracing_export_api_key, flush_traces, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput,ModelSettings # Import necessary classes and functions from the agents module
import os
from pydantic import BaseModel, Field

In [3]:
# Load environment variables from a .env file, overriding existing ones if necessary
load_dotenv(override=True)

True

In [4]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:10]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-PT
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [16]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

## It's easy to use any models with OpenAI compatible endpoints in 3 steps:
### STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [17]:
# Base URLs for different AI services

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/" # Gemini API base URL
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1" # OpenRouter API base URL
GROQ_BASE_URL = "https://api.groq.com/openai/v1" # Groq API base URL

### STEP 2: Create a python client library instance (the async version)

In [18]:
# asyncopenai clients for different AI services
# Gemini client
# OpenRouter client
# Groq client
# OpenAI SDK to use for different AI models 

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

### STEP 3: Create a model object

In [19]:
# OpenAIChatCompletionsModel instances for different AI models
# Gemini model
# Kimi model
# OSS model
# OpenAIChatCompletionsModel use for interacting with different AI models
# Each instance is configured with a specific model and the corresponding OpenAI client.

gemini_model = OpenAIChatCompletionsModel(model="gemini-3.5-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [20]:
# Sales agents for different AI models
# Each agent is configured with a specific model and a set of instructions for writing sales emails.
# here we need to give model with client and model name. unlike the previous setup where the model was directly passed to the agent with OpenAI

sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model,model_settings=ModelSettings(max_tokens=5000),) # Set maximum tokens for the Kimi model
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [21]:
# verify the results of the sales agents
# test each agent are working correctly
result1 = await Runner.run(sales_agent1, instructions)
result2 = await Runner.run(sales_agent2, instructions)
result3 = await Runner.run(sales_agent3, instructions)

In [22]:
# print the final outputs of the sales agents
print(result1.final_output)
print(result2.final_output)
print(result3.final_output)

Understood! As a sales agent for ComplAI, I’m ready to craft persuasive, high-converting emails that highlight how our AI-powered SaaS platform takes the pain out of SOC 2 compliance and audit preparation. 

Whenever you're ready, give me the details of the lead or scenario (e.g., target industry, company size, pain points, or persona), and I'll write a tailored email for you.
 Got it. If you need me to draft a specific outreach sequence, I’ll need a few details:

- **Who** are we emailing? (CTO, CISO, VP Engineering, Compliance Lead, CEO?)
- **What’s their company profile?** (Seed/Series A B2B SaaS, enterprise, fintech, healthcare?)
- **What’s the trigger?** (Cold outbound, post-demo follow-up, pricing objection, post-audit fatigue?)

In the meantime, here are **two proven, high-response templates** tailored to different buyers:

---

### **Template 1: The Technical Leader**  
*(CTO / VP Engineering)*  
**Subject:** SOC2 prep without the sprint-killer

Hi {{Name}},

I keep hearing the

In [23]:
# converted agents as tools for writing sales emails
# Tools for sales agents to write sales emails
# Each tool is created from a sales agent and provides a way to interact with the agent for writing sales emails.

description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [24]:
# Messaging utility for sending sales emails or push notifications
# This utility provides a function to send messages either via email or push notifications based on the USE_EMAIL flag.

# Import the send_email and push functions from the messenger module (messenger.py)
from messenger import send_email, push

# Flag to determine whether to use email or push notifications for sending messages
# Set to True to use email, False to use push notifications
USE_EMAIL = True

# Function to send a message either via email or push notifications based on the USE_EMAIL flag
# Parameters:
#   subject (str): The subject of the message.
#   text_body (str): The plain text body of the message.
#   html_body (str): The HTML body of the message.
# Returns:
#   None
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [25]:
# Test sending a message using the send_message function
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [26]:
# Function tool to send an email using the send_message function
# This function tool wraps the send_message function to provide a convenient way to send emails.
# Agent would use this function tool to send emails to sales prospects.
# The function tool returns a string indicating the success of the email sending operation.

@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [27]:
# List of tools available for the agent to use, including the send_email_tool function.
# The list includes other tools (tool1, tool2, tool3) and the send_email_tool function.
# tool1, tool2, tool3 are agents
# send_email_tool is a function tool for sending emails

tools = [tool1, tool2, tool3, send_email_tool]

In [28]:
# Define the instructions and task for the Sales Manager agent
# The instructions and task define the behavior and responsibilities of the Sales Manager agent.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

# Instantiate the Sales Manager agent with the defined instructions, tools, and model.
# if you notice, model is specified as "gpt-5.4-mini" for the Sales Manager agent.
## gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
## sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
# earlier in above two line, we passed the model as an object because those agents were using different model instances (gemini_model from Google).
set_tracing_disabled(False)
set_tracing_export_api_key(openai_api_key)
sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")

In [29]:
# Run the Sales Manager agent across different models and capture the final output.
# This code runs the Sales Manager agent with the defined task and captures the final output.
# process :
# 1. The Sales Manager agent is run with the defined task.
# 2. The agent generates email drafts using the sales_agent tools.
# 3. The agent evaluates the drafts and selects the best one.
# 4. The best email is sent using the send_email_tool.
# 5. The final output is captured and printed.

with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

Sent the best email.


## Check out the trace
https://platform.openai.com/traces

# Part 2: Structured Outputs
An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object
3. The LLM outputs JSON, and the framework populates a Python object based on it
When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.

In [30]:
# what is structure for reviewing emails
# The structure includes three main aspects: professionalism, number of sentences, and presence of placeholders.
# This structure helps ensure that emails are professional, concise, and personalized.
# The EmailReview class below defines this structure.

# BaseModel is used as the base class for the EmailReview class.
# pydantic BaseModel is used to define the structure of the email review.
# what is pydantic BaseModel?
# pydantic BaseModel is a base class provided by the Pydantic library that allows you to define the structure of your data using Python classes.
# It helps with data validation, settings management, and creating models for complex data structures.

class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [ ]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [101]:
# Example email for review
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

DM
"""

In [102]:
# Create an agent to review the email and run it through the checker.
# The agent will analyze the email based on the EmailReview structure and provide a result.
# agent will check the professionalism, number of sentences, and presence of placeholders in the email.

checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [103]:
# Extract the final review output from the result object.
# This will give us the structured review of the email based on the EmailReview class.
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [104]:
review.is_professional

False

# Part 3: Guardrails
Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/

# Watch out

There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.

In [105]:
# explain the email guardrail function
# This function acts as a guardrail for email content.
# It runs the email through the checker agent, retrieves the review, and determines if there are any issues.
# The function returns a GuardrailFunctionOutput with the review and a flag indicating if a problem was detected.

# parameters:
# ctx: The context object containing the current execution context.
# agent: The agent instance used to run the email through the checker.
# message: The email content to be reviewed.

# returns:
# GuardrailFunctionOutput: An object containing the review and a flag indicating if a problem was detected.

@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [108]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [110]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

"Well howdy there, partner! Sounds like you're lookin' to wrangle yourself some compliance business without gettin' bucked off your horse. \n\nNow, I reckon I could just whip somethin' up right out of my own saddlebag, but a smart trail boss tests his gear before headin' into the badlands. Let me rustup them `sales_agent` tools and see which cold email packs the truest punch for your herd. \n\nHold your hosses for just one second while I check the arsenal!"

Check out the trace:

https://platform.openai.com/traces

# On the other hand..
To state the obvious, this is simpler and will work in any framework

In [111]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Well howdy, partner! Step on up to the corral, 'cause I reckon you've got your hands full wrangling compliance beasts out on the wild frontier of business. 

At ComplAI, we ain't here to beat around the cactus. We got ourselves a top-tier automated platform that rounds up SOC 2, HIPAA, and ISO 27002 compliance faster than a wild stallion on an open prairie. No more losin' sleep wonderin' if an audit's gonna buck you off your saddle. 

How's about we saddle up for a quick 10-minute chat this Tuesday or Thursday? Let me show you how to tame that compliance monster once and for all.

What do ya say, podner? Ready to ride easy?

Best trails,

Sales Manager  
ComplAI


In [112]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent


# OPTIONAL EXTRA: Sandbox Agents
This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:

1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [7]:
%pip install "openai-agents[docker]"

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.docker  import (
    DockerSandboxClient,
    DockerSandboxClientOptions,
)
import docker

In [6]:
CODE_DIR = Path(r"C:\code\AIAgents\2_openai\code")
OUTPUT_DIR = Path(r"C:\code\AIAgents\2_openai\output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [7]:
CODE_DIR

WindowsPath('C:/code/AIAgents/2_openai/code')

In [8]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [9]:
OUTPUT_DIR = Path("output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = Manifest(
    root="/workspace",
    entries={"code": LocalDir(src=CODE_DIR)},
    extra_path_grants=[
        SandboxPathGrant(
            path="/output",
            host_path=str(OUTPUT_DIR),
            read_only=False,
        )
    ],
)
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [10]:
run_config = RunConfig(
    sandbox=SandboxRunConfig(
        client=DockerSandboxClient(docker.from_env()),
        options=DockerSandboxClientOptions(
            image="python:3.12-slim",
        ),
    ),
    workflow_name="Sandbox coding example",
)

In [11]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [12]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

I couldn’t find any source files in `/workspace/code` to inspect or patch, so I wasn’t able to fix the bug yet.

If you want, point me at the correct project folder or add the files there, and I’ll patch it and write the fixed version to `C:\code\AIAgents\2_openai\output`.


# OPTIONAL EXTRA: MCP Teaser!

In [31]:
# Import the MCPServerStreamableHttp class from the agents.mcp module
from agents.mcp import MCPServerStreamableHttp

In [32]:
# Create an agent and run a task with it
# Import the Agent and Runner classes from the agents module

agent = Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini")
result = await Runner.run(agent, task)
print(result.final_output)

I can't directly use sales_agent tools to generate email drafts. However, I can help you create a sales email draft if you provide some context or details about the product or service you want to promote. Would you like to proceed with that?


In [34]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model="gpt-4o-mini")
    result = await Runner.run(agent, task)

print(result.final_output)

### Email Drafts

Here are the drafts generated from the different sales agent tools:

#### Draft 1: From Sales Skills
**Subject:** Quick question about enhancing your sales strategy

Hi [Name],

I noticed that your company is growing rapidly, and I believe we could help you accelerate that growth even further. We specialize in optimizing sales processes for companies like yours, enabling them to achieve greater efficiency and success.

Would you have 15 minutes this week to discuss how we can assist you?

Best regards,  
[Your Name]

---

#### Draft 2: From Dynamics 365 Sales
**Subject:** Ideas to Enhance Your Sales Funnel

Hi [Name],

I’ve been following your recent success in the market, and I have some thoughts on how our tools can help you streamline your sales funnel. Our analytics and engagement tools are designed to enhance customer relationships and close deals faster.

Are you available this week for a quick chat?

Best,  
[Your Name]

---

#### Draft 3: From GojiberryAI Sale